# DeepJSCC-Q paper-based implementation: DIV2K → Kodak

Instrumented Kaggle run. This notebook implements the paper's finite-constellation DeepJSCC idea because an author-linked standalone implementation was not found in the audit. It trains a convolutional image JSCC autoencoder with a learned, power-normalized 16-point complex constellation and straight-through nearest-neighbor quantization. It trains independently for AWGN and Rayleigh at 10 dB for 150 epochs and evaluates Kodak-24.

**Reproduction boundary:** this is a paper-based reimplementation, not an official code reproduction. It uses the same 1/6 complex-symbol rate as the prior analog baselines, 128×128 DIV2K crops, batch 32, and records exact run configs. Rayleigh equalization assumes perfect receiver CSI.

Paper: [DeepJSCC-Q (IEEE JSAIT 2022)](https://arxiv.org/abs/2206.08100). No official standalone repository was identified in the paper/author-source audit.


Compatibility note: the author-linked base encoder squeezes the leading batch axis when batch size is 1; this notebook restores that axis before quantization/decoding without changing latent values.

In [ ]:
import subprocess, sys, time
print('KAGGLE BOOT: DeepJSCC-Q run started at', time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()), flush=True)
print('STARTUP: installing metric dependency (300 second timeout)...', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pytorch-msssim'],check=True,timeout=300)
print('STARTUP: metric dependency ready.', flush=True)


In [ ]:
from pathlib import Path
import os, subprocess, sys, json, time, random, csv, math
print('STARTUP: importing NumPy, Pillow, and PyTorch...', flush=True)
import numpy as np
from PIL import Image
import torch
print('STARTUP: PyTorch imported.', flush=True)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
print('STARTUP: checking CUDA devices...', flush=True)
assert torch.cuda.is_available(), 'Enable Kaggle GPU T4 x2.'
print('GPU(s):',torch.cuda.device_count(),[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)
try:
    subprocess.run(['nvidia-smi'],check=False,timeout=10)
except subprocess.TimeoutExpired:
    print('nvidia-smi timed out after 10s; continuing.',flush=True)
print('STARTUP: checking mounted DIV2K and Kodak inputs...',flush=True)
DIV2K=Path('/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Training/HR/hr_images')
KODAK=Path('/kaggle/input/datasets/sherylmehta/kodak-dataset')
TRAIN_FILES=sorted(p for p in DIV2K.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg'})
KODAK_FILES=sorted(KODAK.rglob('kodim*.png'))
assert len(TRAIN_FILES)==800 and len(KODAK_FILES)==24,(len(TRAIN_FILES),len(KODAK_FILES))
print('DIV2K:',len(TRAIN_FILES),'Kodak:',len(KODAK_FILES),'GPUs:',torch.cuda.device_count(),flush=True)
SRC=Path('/kaggle/working/deep-jscc-pytorch')
if not SRC.exists():
    print('STARTUP: cloning author-linked DeepJSCC base repo...',flush=True)
    subprocess.run(['git','clone','--depth','1','https://github.com/chunbaobao/Deep-JSCC-PyTorch.git',str(SRC)],check=True,timeout=300)
    print('STARTUP: source clone complete.',flush=True)
REV=subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'],text=True).strip()
sys.path.insert(0,str(SRC))
from model import _Encoder,_Decoder
print('Base architecture commit:',REV,flush=True)
OUT=Path('/kaggle/working/seven_model_results/DeepJSCC_Q'); OUT.mkdir(parents=True,exist_ok=True)
SEED=20261018; EPOCHS=150; CROP=128; BATCH=32; C=8; M=16; SNR_DB=10.0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

class CropSet(Dataset):
    def __len__(self): return len(TRAIN_FILES)
    def __getitem__(self,i):
        im=Image.open(TRAIN_FILES[i]).convert('RGB'); w,h=im.size
        x=random.randrange(w-CROP+1); y=random.randrange(h-CROP+1); im=im.crop((x,y,x+CROP,y+CROP))
        if random.random()<.5: im=im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        return torch.from_numpy(np.asarray(im,dtype=np.float32).transpose(2,0,1).copy()/255.)

class ConvJSCCQ(nn.Module):
    def __init__(self,c=C,m=M):
        super().__init__(); self.c=c; self.m=m
        self.encoder=_Encoder(c=c); self.decoder=_Decoder(c=c)
        q=int(math.sqrt(m)); grid=torch.arange(-(q-1),q,2,dtype=torch.float32); re,im=torch.meshgrid(grid,grid,indexing='ij'); pts=torch.stack([re.flatten(),im.flatten()],-1)[:m]; self.constellation=nn.Parameter(pts)
    def normalized_constellation(self):
        p=self.constellation-self.constellation.mean(0,keepdim=True); return p/torch.sqrt(p.square().sum(-1).mean().clamp_min(1e-8))
    def forward(self,x,channel):
        z=self.encoder(x)
        # The author-linked encoder squeezes dimension 0 for a single-item batch; restore it for profiler/Kodak inference.
        if z.ndim==3: z=z.unsqueeze(0)
        b,_,h,w=z.shape; zr,zi=z[:,:self.c],z[:,self.c:]
        power=(zr.square()+zi.square()).mean(dim=(1,2,3),keepdim=True).sqrt().clamp_min(1e-8)
        zc=torch.stack([zr/power,zi/power],dim=-1)
        points=self.normalized_constellation()
        flat=zc.permute(0,2,3,1,4).reshape(b,-1,2)
        d=(flat.square().sum(-1,keepdim=True)-2*flat@points.T+points.square().sum(-1)[None,:])
        idx=d.argmin(-1); q=points[idx].reshape(b,h,w,self.c,2).permute(0,3,1,2,4)
        # Forward uses the nearest learned finite constellation point; STE carries gradients to source encoder and constellation.
        q=q+zc-zc.detach()
        tx=torch.stack([q[...,0],q[...,1]],dim=1).reshape(b,2*self.c,h,w)
        if channel=='AWGN':
            sigma=math.sqrt(1/(2*10**(SNR_DB/10))); rx=tx+torch.randn_like(tx)*sigma
        else:
            hr=torch.randn((b,1,1,1),device=x.device)/math.sqrt(2); hi=torch.randn_like(hr)/math.sqrt(2)
            # One quasi-static complex fade per image; ideal coherent equalization at receiver.
            a=tx[:,:self.c]; bq=tx[:,self.c:]; yr=hr*a-hi*bq; yi=hi*a+hr*bq
            sigma=math.sqrt(1/(2*10**(SNR_DB/10))); yr=yr+torch.randn_like(yr)*sigma; yi=yi+torch.randn_like(yi)*sigma
            den=(hr.square()+hi.square()).clamp_min(1e-4); rx=torch.cat([(hr*yr+hi*yi)/den,(hr*yi-hi*yr)/den],1)
        return self.decoder(rx).clamp(0,1), idx

loader=DataLoader(CropSet(),batch_size=BATCH,shuffle=True,num_workers=4,pin_memory=True,drop_last=True,persistent_workers=True)
print('Steps/epoch:',len(loader),'samples/epoch:',len(loader)*BATCH,'complex-use CBR:',C/(16*3),flush=True)


In [ ]:
from torch.profiler import profile, ProfilerActivity
import pandas as pd

def profile_model(model,channel):
    print(f'{channel}: starting FLOPs profile...',flush=True)
    x=torch.rand(1,3,CROP,CROP,device='cuda:0'); model.train(); model.zero_grad(set_to_none=True)
    with profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA],with_flops=True) as p:
        rec,_=model(x,channel); F.mse_loss(rec,x).backward()
    tr=sum(int(e.flops or 0) for e in p.key_averages()); model.zero_grad(set_to_none=True); model.eval()
    with torch.no_grad(),profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA],with_flops=True) as p: model(x,channel)
    inf=sum(int(e.flops or 0) for e in p.key_averages()); print(f'{channel}: FLOPs profile complete; train={tr}, inference={inf}',flush=True); return tr,inf

def quality(a,b):
    from pytorch_msssim import ssim,ms_ssim
    mse=F.mse_loss(a,b).item(); return 10*math.log10(1/max(mse,1e-12)),ssim(a,b,data_range=1.,size_average=True).item(),ms_ssim(a,b,data_range=1.,size_average=True).item()

def train_channel(channel):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED); torch.cuda.manual_seed_all(SEED)
    run=OUT/channel; run.mkdir(parents=True,exist_ok=True); print(f'{channel}: constructing DeepJSCC-Q model...',flush=True); model=ConvJSCCQ().to('cuda:0')
    if torch.cuda.device_count()>1:
        model=nn.DataParallel(model,device_ids=list(range(torch.cuda.device_count())))
        print('Using DataParallel on',torch.cuda.device_count(),'T4 GPUs.',flush=True)
    else: print('Using one visible GPU.',flush=True)
    opt=torch.optim.Adam(model.parameters(),lr=1e-4)
    tr_flops,inf_flops=profile_model(model,channel); rows=[]
    for ep in range(1,EPOCHS+1):
        model.train(); t=time.time(); loss_sum=0.; n=0
        for batch_idx,x in enumerate(loader,1):
            x=x.to('cuda:0',non_blocking=True); opt.zero_grad(set_to_none=True); rec,_=model(x,channel); loss=F.mse_loss(rec,x); loss.backward(); opt.step(); loss_sum+=float(loss.detach())*len(x); n+=len(x)
            if batch_idx==1 or batch_idx%5==0 or batch_idx==len(loader): print(f'{channel} epoch {ep}/{EPOCHS} batch {batch_idx}/{len(loader)} loss={float(loss.detach()):.6f} elapsed={time.time()-t:.1f}s',flush=True)
        row={'model':'DeepJSCC-Q paper-based','channel':channel,'snr_db':SNR_DB,'epoch':ep,'train_mse':loss_sum/max(n,1),'epoch_seconds':time.time()-t,'samples_seen':n,'div2k_pool_images':len(TRAIN_FILES),'crop_px':CROP,'batch_size':BATCH,'encoder_decoder_source_commit':REV,'complex_channel_uses_per_pixel_rate':C/48,'constellation_size':M,'training_flops_per_sample':tr_flops,'inference_flops_per_image_128':inf_flops}
        rows.append(row); pd.DataFrame(rows).to_csv(run/'epoch_metrics.csv',index=False)
        tmp=run/'model.pth.tmp'; torch.save({'epoch':ep,'model':model.state_dict(),'optimizer':opt.state_dict(),'config':row},tmp); tmp.replace(run/'model.pth')
        print('EPOCH COMPLETE',json.dumps(row),flush=True)
    per=[]; model.eval()
    with torch.no_grad():
        for path in KODAK_FILES:
            im=Image.open(path).convert('RGB'); a=np.asarray(im,dtype=np.float32)/255.; h,w=a.shape[:2]; h0=h-h%16; w0=w-w%16
            ref=torch.from_numpy(a[:h0,:w0].transpose(2,0,1).copy()).unsqueeze(0).to('cuda:0'); rec,idx=model(ref,channel); ps,ss,ms=quality(rec,ref); uses=C*(h0//4)*(w0//4)
            per.append({'model':'DeepJSCC-Q paper-based','channel':channel,'snr_db':SNR_DB,'image':path.stem,'height':h0,'width':w0,'psnr_db':ps,'ssim':ss,'msssim':ms,'complex_channel_uses':uses,'cbr':uses/(3*h0*w0),'source_bits_per_symbol':math.log2(M),'training_flops_per_sample':tr_flops,'inference_flops_per_image_128':inf_flops,'source_commit':REV})
    pdf=pd.DataFrame(per); pdf.to_csv(run/'kodak_per_image.csv',index=False); print(f'{channel}: evaluated {len(per)} Kodak images.',flush=True)
    summary={'model':'DeepJSCC-Q paper-based','channel':channel,'snr_db':SNR_DB,'epochs':EPOCHS,'kodak_psnr_db':pdf.psnr_db.mean(),'kodak_ssim':pdf.ssim.mean(),'kodak_msssim':pdf.msssim.mean(),'mean_complex_channel_uses_per_image':pdf.complex_channel_uses.mean(),'mean_cbr':pdf.cbr.mean(),'constellation_size':M,'training_flops_per_sample':tr_flops,'inference_flops_per_image_128':inf_flops,'source_bits_per_symbol':math.log2(M)}
    pd.DataFrame([summary]).to_csv(run/'summary.csv',index=False); print('CHANNEL SUMMARY',json.dumps(summary),flush=True); del model,opt; torch.cuda.empty_cache()

for ch in ['AWGN','Rayleigh']: train_channel(ch)
print('ALL DeepJSCC-Q channels finished:',OUT,flush=True)
